# 04. 다중회귀 — 점진적 변수 추가 모델 & 조정지표

| 모델 | 추가 변수 그룹 |
|------|---------------|
| M0 (기본) | living_pop, food_accom + 연도 FE |
| M1 | + 주거 특성 (elderly_ratio, single_hh_ratio) |
| M2 | + 상업/활동 (emp_per_pop) |
| M3 | + 관광/방문/상권 밀집 (day_night_ratio) |
| M4 | + 소비 (sales_per_pop) |
| M_full | M0~M4 전체 통합 |
| M5 | VIF 기반 자동 제거 (M4 출발) |
| M6 | M5 + 물류/도소매 (강서구 이상치 보완) |
| M7 | M6 + 외국인비율/지하철역수/대형사업체 (용산구 이상치 보완) |

**종속변수**: log(waste_total)  /  **모든 연속변수 표준화**  /  **패널 n=100**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

BASE  = Path('../../data')
PROC  = BASE / 'processed'
FINAL = BASE / 'final'

COL_KR = {
    'living_pop_std':          '생활인구 일평균',
    'food_accom_std':          '음식숙박업 수',
    'elderly_std':             '고령인구 비율',
    'single_hh_std':           '1인가구 비율',
    'emp_per_pop_std':         '종사자/인구',
    'day_night_std':           '주야간 인구비',
    'sales_per_pop_std':       '1인당 매출',
    'transport_per_pop_std':   '운수업 사업체/인구',
    'retail_per_pop_std':      '도소매업 사업체/인구',
    'foreigner_ratio_std':     '외국인 생활인구 비율',
    'subway_per_pop_std':      '지하철역 수/인구',
    'large_biz_per_pop_std':   '대형사업체(100인+)/인구',
}
print('설정 완료')

## 1. 데이터 로드 및 변수 파생

In [ ]:
def load_year_all(path):
    return pd.read_csv(path)

# eda_master (기본 변수)
df = pd.read_csv(FINAL / 'eda_master_v2_by_gu_year.csv', encoding='utf-8-sig')

# 1인가구 비율
df_hh = load_year_all(PROC / 'population' / 'household_by_gu_year.csv')
df = df.merge(df_hh[['year','gu','single_hh_ratio']], on=['year','gu'], how='left')

# 종사자 수 + 물류(운수업) + 도소매업
df_biz = load_year_all(PROC / 'business' / 'business_by_industry_gu_year.csv')
df = df.merge(
    df_biz[['year','gu','emp_total','transport_biz','retail_biz']],
    on=['year','gu'], how='left'
)

# 매출
df_sale = load_year_all(PROC / 'business' / 'sales_by_gu_year.csv')
df = df.merge(df_sale[['year','gu','annual_sales']], on=['year','gu'], how='left')

# 외국인 생활인구 (1순위 — 용산구 설명)
df_lp = load_year_all(PROC / 'living_population' / 'living_pop_merged_by_gu_year.csv')
df = df.merge(df_lp[['year','gu','foreigner_lp_daily_avg']], on=['year','gu'], how='left')

# 지하철역 수 (2순위 — 역 수 proxy; 실제 승하차 데이터 없음)
df_att = load_year_all(PROC / 'business' / 'attraction_by_gu_year.csv')
df = df.merge(df_att[['year','gu','subway_cnt']], on=['year','gu'], how='left')

# 대형사업체(종사자 100인 이상) (3순위 — 대형상업시설 proxy)
df_sz = load_year_all(PROC / 'business' / 'business_by_size_gu_year.csv')
df_sz['large_biz'] = df_sz['s100_299_biz'] + df_sz['s1000plus_biz']
df = df.merge(df_sz[['year','gu','large_biz']], on=['year','gu'], how='left')

# 파생변수
df['log_waste']          = np.log(df['waste_total'])
df['emp_per_pop']        = df['emp_total']              / df['total_pop']
df['sales_per_pop']      = df['annual_sales']           / df['total_pop']
df['transport_per_pop']  = df['transport_biz']          / df['total_pop']
df['retail_per_pop']     = df['retail_biz']             / df['total_pop']
df['foreigner_ratio']    = df['foreigner_lp_daily_avg'] / df['living_pop_daily_avg']
df['subway_per_pop']     = df['subway_cnt']             / df['total_pop']
df['large_biz_per_pop']  = df['large_biz']              / df['total_pop']

print('shape:', df.shape)
print('연도:', sorted(df['year'].unique()))
check_cols = ['log_waste','living_pop_daily_avg','food_accom_biz',
              'elderly_ratio','single_hh_ratio','emp_per_pop',
              'day_night_ratio','sales_per_pop',
              'transport_per_pop','retail_per_pop',
              'foreigner_ratio','subway_per_pop','large_biz_per_pop']
missing = df[check_cols].isnull().sum()
print('결측:\n', missing[missing > 0])

## 2. 연속변수 표준화

In [ ]:
RAW_VARS = [
    ('living_pop_daily_avg', 'living_pop_std'),
    ('food_accom_biz',       'food_accom_std'),
    ('elderly_ratio',        'elderly_std'),
    ('single_hh_ratio',      'single_hh_std'),
    ('emp_per_pop',          'emp_per_pop_std'),
    ('day_night_ratio',      'day_night_std'),
    ('sales_per_pop',        'sales_per_pop_std'),
    ('transport_per_pop',    'transport_per_pop_std'),
    ('retail_per_pop',       'retail_per_pop_std'),
    ('foreigner_ratio',      'foreigner_ratio_std'),
    ('subway_per_pop',       'subway_per_pop_std'),
    ('large_biz_per_pop',    'large_biz_per_pop_std'),
]

for raw, std in RAW_VARS:
    scaler = StandardScaler()
    df[std] = scaler.fit_transform(df[[raw]])

STD_VARS = [s for _, s in RAW_VARS]
print('표준화 완료:', STD_VARS)
df[STD_VARS].describe().round(3)

## 3. 모델 피팅 헬퍼 함수

In [ ]:
def calc_vif(df_, vars_):
    X = df_[vars_].dropna().values.astype(float)
    X = (X - X.mean(0)) / (X.std(0) + 1e-10)
    vif_vals = []
    for i in range(X.shape[1]):
        y_ = X[:, i]
        X_ = np.column_stack([np.ones(len(y_)), np.delete(X, i, axis=1)])
        c, *_ = np.linalg.lstsq(X_, y_, rcond=None)
        ss_res = np.sum((y_ - X_ @ c) ** 2)
        ss_tot = np.sum((y_ - y_.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        vif_vals.append(round(1 / (1 - r2) if r2 < 1 else np.inf, 2))
    return pd.DataFrame({
        '변수': [COL_KR.get(v, v) for v in vars_],
        'VIF':  vif_vals,
        '판정': ['⚠>10' if v > 10 else ('△5~10' if v > 5 else '✓<5') for v in vif_vals]
    })

def fit_model(label, cont_vars, df_=df):
    formula = 'log_waste ~ ' + ' + '.join(cont_vars) + ' + C(year)'
    model   = smf.ols(formula, data=df_).fit()

    sep = '=' * 55
    print(sep)
    print(' ' + label)
    print(sep)
    print('  R²=' + f'{model.rsquared:.4f}' +
          '  Adj R²=' + f'{model.rsquared_adj:.4f}' +
          '  F p=' + f'{model.f_pvalue:.3e}' +
          '  CondNum=' + f'{model.condition_number:.1f}')
    print()

    for v in cont_vars:
        c = model.params.get(v, np.nan)
        p = model.pvalues.get(v, np.nan)
        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
        name = COL_KR.get(v, v)
        print(f'  {name:22s}  coef={c:+.4f}  p={p:.4f}  {star}')
    print()

    vif = calc_vif(df_, cont_vars)
    print(vif.to_string(index=False))
    print()

    return model

print('헬퍼 함수 정의 완료')

## 4. M0 — 기본 모델

In [ ]:
VARS_M0 = ['living_pop_std', 'food_accom_std']
m0 = fit_model('M0: 기본 (생활인구 + 음식숙박업)', VARS_M0)

## 5. M1 — M0 + 주거 특성 (elderly_ratio, single_hh_ratio)

In [ ]:
VARS_M1 = VARS_M0 + ['elderly_std', 'single_hh_std']
m1 = fit_model('M1: M0 + 주거 특성 (고령비율, 1인가구비율)', VARS_M1)

## 6. M2 — M1 + 상업/활동 (emp_per_pop)

In [ ]:
VARS_M2 = VARS_M1 + ['emp_per_pop_std']
m2 = fit_model('M2: M1 + 상업/활동 (종사자/인구)', VARS_M2)

## 7. M3 — M2 + 관광/방문/상권 밀집 (day_night_ratio)

In [ ]:
VARS_M3 = VARS_M2 + ['day_night_std']
m3 = fit_model('M3: M2 + 관광/방문 (주야간인구비)', VARS_M3)

## 8. M4 — M3 + 소비 (sales_per_pop)

In [ ]:
VARS_M4 = VARS_M3 + ['sales_per_pop_std']
m4 = fit_model('M4: M3 + 소비 (1인당 매출)', VARS_M4)

## 9. M_full — 전체 통합

In [ ]:
VARS_FULL = list(dict.fromkeys(VARS_M4))
m_full = fit_model('M_full: 전체 통합', VARS_FULL)

## 10. M5 — VIF 기반 점진적 변수 제거 (M4 출발)

> VIF > 5 인 변수를 가장 높은 것부터 하나씩 제거 → 모든 VIF ≤ 5 될 때까지 반복

In [ ]:
def vif_reduce(start_vars, threshold=5.0, df_=df):
    current = start_vars.copy()
    step = 0
    while True:
        vif_df = calc_vif(df_, current)
        max_vif = vif_df['VIF'].max()
        worst_idx = vif_df['VIF'].idxmax()
        worst_var = current[worst_idx]
        worst_nm  = COL_KR.get(worst_var, worst_var)

        print('--- Step ' + str(step) + '  (변수 ' + str(len(current)) + '개) ---')
        print(vif_df[['변수','VIF','판정']].to_string(index=False))

        if max_vif <= threshold:
            print()
            print('  ✓ 모든 VIF <= ' + str(threshold) + '  ->  제거 종료')
            print()
            break

        print()
        print('  -> 제거: ' + worst_nm + '  (VIF=' + f'{max_vif:.2f}' + ')')
        print()
        current.remove(worst_var)
        step += 1

        if len(current) == 1:
            print('  변수가 1개만 남아 종료')
            break

    return current

print('=' * 50)
print(' M4 -> M5 : VIF 기반 변수 제거 과정')
print('=' * 50)
VARS_M5 = vif_reduce(VARS_M4.copy(), threshold=5.0)
print('최종 채택 변수 (' + str(len(VARS_M5)) + '개):', VARS_M5)
print()
m5 = fit_model('M5: VIF 제거 최적 모델', VARS_M5)

## 11. M6 — M5 + 물류/도소매 변수 (강서구 이상치 보완)

> **운수업 사업체/인구** (`transport_per_pop`): 공항·물류창고 밀집 → 강서구
> **도소매업 사업체/인구** (`retail_per_pop`): 대형상업시설 proxy

In [ ]:
VARS_M6_START = VARS_M5 + ['transport_per_pop_std', 'retail_per_pop_std']
print('=' * 50)
print(' M5 -> M6 : 물류/도소매 변수 추가 후 VIF 제거')
print('=' * 50)
VARS_M6 = vif_reduce(VARS_M6_START.copy(), threshold=5.0)
print('최종 채택 변수 (' + str(len(VARS_M6)) + '개):', VARS_M6)
print()
m6 = fit_model('M6: M5 + 물류/도소매 (강서구 보완)', VARS_M6)

## 12. M7 — M6 + 외국인비율/지하철역수/대형사업체 (용산구 이상치 보완)

| 변수 | 원천 | 비고 |
|------|------|------|
| `foreigner_ratio` | 외국인 생활인구 / 전체 생활인구 | 용산구 이태원·외국인 집중 지역 |
| `subway_per_pop` | 지하철역 수 / 등록인구 | 실제 승하차 데이터 없어 역 수로 대체 |
| `large_biz_per_pop` | 100인 이상 사업체 수 / 등록인구 | 대형상업시설·백화점 proxy |

> VIF > 5 이면 자동 제거

In [ ]:
VARS_M7_START = VARS_M6 + [
    'foreigner_ratio_std',
    'subway_per_pop_std',
    'large_biz_per_pop_std',
]
print('=' * 50)
print(' M6 -> M7 : 외국인비율/지하철역/대형사업체 추가 후 VIF 제거')
print('=' * 50)
VARS_M7 = vif_reduce(VARS_M7_START.copy(), threshold=5.0)
print('최종 채택 변수 (' + str(len(VARS_M7)) + '개):', VARS_M7)
print()
m7 = fit_model('M7: M6 + 외국인비율/지하철역/대형사업체 (용산구 보완)', VARS_M7)

## 13. 모델 성능 비교

In [ ]:
models = {
    'M0 (기본)':           (m0,     VARS_M0),
    'M1 (+주거)':          (m1,     VARS_M1),
    'M2 (+상업/활동)':     (m2,     VARS_M2),
    'M3 (+관광/방문)':     (m3,     VARS_M3),
    'M4 (+소비)':          (m4,     VARS_M4),
    'M5 (VIF제거)':        (m5,     VARS_M5),
    'M6 (+물류/도소매)':   (m6,     VARS_M6),
    'M7 (+외국인/지하철)': (m7,     VARS_M7),
    'M_full':              (m_full, VARS_FULL),
}

rows = []
for name, (m, vlist) in models.items():
    max_vif = calc_vif(df, vlist)['VIF'].max()
    rows.append({
        '모델':    name,
        '변수 수': len(vlist),
        'R²':      round(m.rsquared, 4),
        'Adj R²':  round(m.rsquared_adj, 4),
        'AIC':     round(m.aic, 1),
        'BIC':     round(m.bic, 1),
        'CondNum': round(m.condition_number, 1),
        'max VIF': max_vif,
    })

cmp = pd.DataFrame(rows).set_index('모델')
print(cmp.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
model_names = list(models.keys())
r2_vals     = cmp['R²'].values
adjr2_vals  = cmp['Adj R²'].values
aic_vals    = cmp['AIC'].values
maxvif_vals = cmp['max VIF'].values
x = range(len(model_names))

axes[0].bar(x, r2_vals,    color='steelblue', edgecolor='white', label='R²',    alpha=0.7)
axes[0].bar(x, adjr2_vals, color='coral',     edgecolor='white', label='Adj R²',alpha=0.7)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=40, ha='right')
axes[0].set_ylabel('R²'); axes[0].set_title('R² / Adj R² 비교', fontweight='bold')
axes[0].legend()

axes[1].plot(x, aic_vals, 'o-', color='steelblue', linewidth=2)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=40, ha='right')
axes[1].set_ylabel('AIC'); axes[1].set_title('AIC (낮을수록 좋음)', fontweight='bold')

bar_colors = ['tomato' if v > 10 else ('orange' if v > 5 else 'steelblue')
              for v in maxvif_vals]
axes[2].bar(x, maxvif_vals, color=bar_colors, edgecolor='white')
axes[2].axhline(5,  color='orange', linestyle='--', linewidth=1.0, label='주의(5)')
axes[2].axhline(10, color='tomato', linestyle='--', linewidth=1.0, label='위험(10)')
axes[2].set_xticks(x); axes[2].set_xticklabels(model_names, rotation=40, ha='right')
axes[2].set_ylabel('최대 VIF'); axes[2].set_title('다중공선성 (최대 VIF)', fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle('모델 비교', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 14. 최종 모델 선택

> 아래 셀의 `BEST_MODEL`과 `BEST_VARS`를 원하는 모델로 수정하세요.
> 기준: **Adj R² 최대화 + max VIF < 5 + AIC 최소**

In [ ]:
# ── 여기서 최종 모델을 선택 ───────────────────────────────────────
BEST_MODEL = m7
BEST_VARS  = VARS_M7
BEST_LABEL = 'M7 (+외국인/지하철/대형사업체)'
# ──────────────────────────────────────────────────────────────────

fitted    = BEST_MODEL.fittedvalues.values
residuals = BEST_MODEL.resid.values

print('선택 모델:', BEST_LABEL)
print('R²=' + f'{BEST_MODEL.rsquared:.4f}' +
      '  Adj R²=' + f'{BEST_MODEL.rsquared_adj:.4f}')
print('CondNum=' + f'{BEST_MODEL.condition_number:.1f}')
print()
print(BEST_MODEL.summary2())

## 15. 잔차 진단 (최종 모델)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(fitted, residuals, alpha=0.6, s=50,
                color='steelblue', edgecolors='white')
axes[0].axhline(0, color='red', linewidth=1.2, linestyle='--')
axes[0].set_xlabel('예측값 (log)'); axes[0].set_ylabel('잔차')
axes[0].set_title('잔차 vs 예측값')

axes[1].hist(residuals, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('잔차'); axes[1].set_title('잔차 분포')

sm.qqplot(residuals, line='s', ax=axes[2], alpha=0.7)
axes[2].set_title('Q-Q Plot')

plt.suptitle('잔차 진단 - ' + BEST_LABEL, fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('왜도=' + f'{pd.Series(residuals).skew():.3f}' +
      '  첨도=' + f'{pd.Series(residuals).kurt():.3f}')

## 16. 이상치 진단 (Cook's Distance)

In [ ]:
influence  = BEST_MODEL.get_influence()
std_resid  = influence.resid_studentized_internal
cooks_d    = influence.cooks_distance[0]
thresh_cd  = 4 / len(df)

df_diag = df[['year','gu','waste_total']].copy()
df_diag['fitted']    = fitted
df_diag['std_resid'] = std_resid
df_diag['cooks_d']   = cooks_d

outliers = df_diag[
    (df_diag['std_resid'].abs() > 2) | (df_diag['cooks_d'] > thresh_cd)
].sort_values('cooks_d', ascending=False)

print('이상치: ' + str(len(outliers)) + '개 / ' + str(len(df)) + '개')
print(outliers[['year','gu','waste_total','std_resid','cooks_d']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gu_std = df_diag.groupby('gu')['std_resid'].mean().sort_values()
bar_c  = ['tomato' if abs(v) > 2 else 'steelblue' for v in gu_std.values]
axes[0].barh(gu_std.index, gu_std.values, color=bar_c, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].axvline( 2, color='red', linestyle='--', linewidth=1.0, label='±2')
axes[0].axvline(-2, color='red', linestyle='--', linewidth=1.0)
axes[0].set_title('자치구별 평균 표준화 잔차', fontweight='bold')
axes[0].legend(fontsize=8)

sns.boxplot(data=df_diag, x='year', y='std_resid', ax=axes[1],
            palette={str(y): c for y, c in
                     zip([2020,2021,2022,2023],
                         ['#4C72B0','#55A868','#C44E52','#DD8452'])})
axes[1].axhline(0,  color='black', linewidth=0.8, linestyle='--')
axes[1].axhline( 2, color='red', linestyle=':', linewidth=1.0)
axes[1].axhline(-2, color='red', linestyle=':', linewidth=1.0)
axes[1].set_title('연도별 잔차 분포', fontweight='bold')
plt.tight_layout(); plt.show()

## 17. 조정지표 산출

> **조정지표 = 실제 발생량(톤) / exp(예측_log) × 100**

In [ ]:
df['pred_waste_ton']  = np.exp(fitted)
df['residual']        = residuals
df['adjusted_index']  = (df['waste_total'] / df['pred_waste_ton'] * 100).round(2)

idx_by_gu = df.groupby('gu')['adjusted_index'].mean().round(2).sort_values(ascending=False)

print('=== 자치구별 평균 조정지표 ===')
for gu, val in idx_by_gu.items():
    bar  = '▓' * int(abs(val - 100) / 2)
    flag = ' ↑ 초과' if val > 105 else (' ↓ 절약' if val < 95 else '')
    print('  ' + f'{gu:5s}' + '  ' + f'{val:6.1f}' + '  ' + bar + flag)

## 18. 조정지표 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bar_colors = ['tomato' if v > 100 else 'steelblue' for v in idx_by_gu.values]
axes[0].bar(idx_by_gu.index, idx_by_gu.values,
            color=bar_colors, edgecolor='white', width=0.7)
axes[0].axhline(100, color='black', linewidth=1.2, linestyle='--')
axes[0].set_xticklabels(idx_by_gu.index, rotation=45, ha='right')
axes[0].set_ylabel('조정지표')
axes[0].set_title('자치구별 평균 조정지표 (' + BEST_LABEL + ')', fontweight='bold')
for sp in ['top','right']: axes[0].spines[sp].set_visible(False)

top5    = idx_by_gu.head(5).index.tolist()
bottom5 = idx_by_gu.tail(5).index.tolist()
pivot   = df[df['gu'].isin(top5+bottom5)].pivot(
    index='year', columns='gu', values='adjusted_index'
)
for gu in top5+bottom5:
    ls = '-' if gu in top5 else '--'
    axes[1].plot(pivot.index, pivot[gu], marker='o',
                 linewidth=1.8, linestyle=ls, label=gu)
axes[1].axhline(100, color='gray', linewidth=1, linestyle=':')
axes[1].set_xlabel('연도'); axes[1].set_ylabel('조정지표')
axes[1].set_title('조정지표 연도별 추이', fontweight='bold')
axes[1].legend(ncol=2, fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle('조정지표 - ' + BEST_LABEL, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 19. 결과 저장

In [ ]:
out_cols = ['year','gu','waste_total','pred_waste_ton','residual','adjusted_index']
df_out = df[out_cols].sort_values(['year','gu']).reset_index(drop=True)
df_out.to_csv(FINAL / 'adjusted_index_by_gu_year.csv',
              index=False, encoding='utf-8-sig')
print('저장 완료:', df_out.shape)

df23 = df_out[df_out['year']==2023].sort_values('adjusted_index', ascending=False)
print()
print('[2023] 상위 5구 (초과 발생):')
print(df23.head(5)[['gu','waste_total','pred_waste_ton','adjusted_index']].to_string(index=False))
print()
print('[2023] 하위 5구 (절약 발생):')
print(df23.tail(5)[['gu','waste_total','pred_waste_ton','adjusted_index']].to_string(index=False))